In this notebook I will create a training table equivalent to ml_model_run_details

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid

import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\application_default_credentials.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


### Function

#### expand_calc_features

In [2]:
import pandas as pd
import json

def expand_calc_features(df):
    """
    Expand the calcFeatures JSON column into separate columns and return the complete DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_list = []

    for idx, calc_features_str in enumerate(df['calcFeatures']):
        try:
            # Parse the JSON string
            features_dict = json.loads(calc_features_str.replace("'", '"'))  # Replace single quotes with double quotes for valid JSON
            calc_features_list.append(features_dict)
        except (json.JSONDecodeError, AttributeError) as e:
            # If parsing fails, create an empty dict and print warning
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            calc_features_list.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_list)

    # Add prefix to JSON-derived columns to avoid conflicts
    calc_features_df = calc_features_df.add_prefix('calc_')

    # Reset index to ensure proper alignment
    df_expanded = df_expanded.reset_index(drop=True)
    calc_features_df = calc_features_df.reset_index(drop=True)

    # Combine original DataFrame with expanded calcFeatures
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    return result_df


#### expand_calc_features_robust

In [3]:
import pandas as pd
import json

def expand_calc_features_robust(df):
    """
    Expand the calcFeatures JSON column into separate columns with better error handling.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_data = []

    for idx, row in df.iterrows():
        calc_features_str = row['calcFeatures']

        if pd.isna(calc_features_str) or calc_features_str == '':
            calc_features_data.append({})
            continue

        try:
            # Clean the string and parse JSON
            cleaned_str = calc_features_str.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            features_dict = json.loads(cleaned_str)
            calc_features_data.append(features_dict)
        except Exception as e:
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            print(f"Problematic string: {calc_features_str[:100]}...")  # Print first 100 chars
            calc_features_data.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_data)

    # Add prefix to JSON-derived columns to avoid conflicts with existing columns
    calc_features_df = calc_features_df.add_prefix('feat_')

    # Combine DataFrames
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    print(f"Original DataFrame shape: {df.shape}")
    print(f"Expanded DataFrame shape: {result_df.shape}")
    print(f"Added {len(calc_features_df.columns)} new columns from calcFeatures")

    return result_df

# Table Name

In [4]:
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"

# transform_datav2

In [5]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_datav2(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v2",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],  # Device operating system
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


# SIL V2

*   Sil-Alpha-CIC-SIL-Model: worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930
*   Sil-Alpha-StackingModel: worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3
*   Sil-Beta-AppsScoreModel:
*   New Applicants (T1, T2): risk_mart.applied_sil_trench1_trench2_loan_jan2024_30sep2025_app_scored, risk_mart.applied_sil_new_applicants_loan_oct0125_oct1325_app_scored
*   Repeat Applicants (T3): risk_mart.b_score_model_applied_loans_sil_20240101_20250930_app_scored, risk_mart.applied_sil_repeat_applicants_loan_oct0125_oct1325_app_scored
*   Sil-Beta-DemoScoreModel: worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015
*   Sil-Beta-StackScoreModel: worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3m

##### 'Alpha - CIC-SIL-Model'

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;
"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
 date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """
delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;
"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Alpha Sil Stack Model 

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models')
print(f"The shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta SIL STACK Score Model

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta Sil App Score

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models')
print(f"the shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

In [ ]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.cb_score ml_score,
r.dl_score dl_score,
app_cnt_productivity_ever,
app_cnt_rated_for_3plus_ever, 
app_cnt_books_and_reference_ever,
app_cnt_tools_ever, 
app_median_time_bw_installed_mins_3d,
app_median_time_bw_installed_mins_30d,
app_cnt_communication_ever, 
app_cnt_finance_90d,
app_cnt_absence_tag_180d, 
app_cnt_shopping_ever,
app_cnt_social_ever, 
app_cnt_driver_90d, 
app_cnt_payday_365d,
app_cnt_driver_365d, 
app_cnt_music_and_audio_ever,
app_cnt_finance_180d, 
app_cnt_art_and_design_ever,
app_cnt_gaming_90d, 
app_avg_time_bw_installed_mins_30d,
app_cnt_education_ever,
case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from risk_mart.applied_sil_repeat_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_productivity_ever', 'app_cnt_rated_for_3plus_ever',
       'app_cnt_books_and_reference_ever', 'app_cnt_tools_ever',
       'app_median_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_30d', 'app_cnt_communication_ever',
       'app_cnt_finance_90d', 'app_cnt_absence_tag_180d',
       'app_cnt_shopping_ever', 'app_cnt_social_ever', 'app_cnt_driver_90d',
       'app_cnt_payday_365d', 'app_cnt_driver_365d',
       'app_cnt_music_and_audio_ever', 'app_cnt_finance_180d',
       'app_cnt_art_and_design_ever', 'app_cnt_gaming_90d',
       'app_avg_time_bw_installed_mins_30d', 'app_cnt_education_ever',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta SIL Demo Score

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 3'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

# 💎 Cash V2

## Ⓜ️ alpha_stack_model_cash

### ₸ Trench 1

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 1'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

### ₸ Trench 2

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 2'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

### ₸ Trench 3

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 3'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

## Ⓜ️ beta_stack_model_cash

### ₸ Trench 1

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 1'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

### ₸ Trench 2

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 2'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

### ₸ Trench 3

#### ₫ Delete

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 3'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

## Ⓜ️ Alpha Cash CIC Model

### ₸ Trench 1

#### ₫ Delete

In [24]:
sq = f"""
DELETE FROM {table_id}
WHERE modelDisplayName = 'Alpha-Cash-CIC-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")



Deleted 132820 rows.


In [25]:
sq = """
with base as 
(SELECT
  lmt.customerId as customer_id,
  r.`digitalLoanAccountId`,
  r.`target_fspd30_flag`,
  `dt_loan_appln_dtime` as application_date,
  alt.`ln_disb_dtime`,
  alt.`ln_loan_type`,
  alt.`ln_loan_level_user_type`,
  CASE
        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) <= 30
        THEN 'Trench 1'

        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) > 30
        THEN 'Trench 2'

        WHEN alt.ln_loan_level_user_type = '1_Repeat Applicant'
        THEN 'Trench 3'

        ELSE NULL
    END AS trench_category,
  r.`year_month`,
  r.`ln_os_type` as osType,
  `ScoreRange_WOE`,
  `CreditAvgCreditLimit`,
  `cnt_active_contracts_Credit_Lines`,
  `vel_contract_nongranted_cnt_6on12`,
  `cnt_active_contracts_Credit_Cards`,
  `avg_amt_granted_24M`,
  `cic_total_overdue_to_total_outstanding_ratio`,
  `cic_total_overdue_to_monthly_income_ratio`,
  `CreditMaxOverdueDays`,
  `cic_seq_share_credit_lines_24m`,
  `Credit_Lines_active_contracts_util`,
  `Personal_Loans_granted_contracts_cnt_6M`,
  `has_ever_been_overdue`,
  `cic_seq_cnt_credit_cards_24m`,
  `has_ever_had_risky_contract`,
  `creditcard_utilization_ever`,
  `cic_seq_first_loan_type_24m`,
  `total_overdue_granted_contracts`,
  `creditcard_utilization_combined`,
  `cic_score`,
  case when date(r.dt_loan_appln_dtime) >= '2026-01-01' and date(r.dt_loan_appln_dtime) <= '2026-04-30' then 'Dev_Train'
           when date(r.dt_loan_appln_dtime) > '2026-04-30' and  date(r.dt_loan_appln_dtime) < '2026-10-30'then 'Dev_Test'
           when date(r.dt_loan_appln_dtime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20250101_20261027_score_range_fix` r
Left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
Left join `prj-prod-dataplatform.risk_mart.applied_loans_trans` alt on alt.digitalLoanAccountId = r.digitalLoanAccountId
where  r.`cic_score` is not null 
)
select * from base 
where trench_category = 'Trench 1'
and  date(application_date) < '2026-10-30' 
and target_fspd30_flag is not null
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 7fc452cf-5ef5-4db6-af22-50e669b10aa2 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (37560, 31)


In [26]:
feature_column = ['ScoreRange_WOE', 'CreditAvgCreditLimit',   'cnt_active_contracts_Credit_Lines', 'vel_contract_nongranted_cnt_6on12', 'cnt_active_contracts_Credit_Cards', 'avg_amt_granted_24M', 'cic_total_overdue_to_total_outstanding_ratio',
       'cic_total_overdue_to_monthly_income_ratio', 'CreditMaxOverdueDays',  'cic_seq_share_credit_lines_24m', 'Credit_Lines_active_contracts_util', 'Personal_Loans_granted_contracts_cnt_6M', 'has_ever_been_overdue',
       'cic_seq_cnt_credit_cards_24m', 'has_ever_had_risky_contract', 'creditcard_utilization_ever', 'cic_seq_first_loan_type_24m',  'total_overdue_granted_contracts', 'creditcard_utilization_combined', 'cic_score']

dfd = transform_datav2(data, feature_column, a='cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 1', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,4137509,805007d4-6d39-4c8e-b732-1bf602efb9d9,913ff31d-1f99-4db3-b5c2-ba68b9c78362,0.239757,2026-11-19T08:29:50.840433,2026-11-19T08:29:50.840433,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": -0.27448585216942006, ""Cred...",Cash Stack v2 Models,595518a1-72b5-4a57-90fc-85ed1144ca0e,2026-11-19T08:29:50.840433,{},Trench 1,Android,Dev_Train,2026-02-24 11:02:47
1,3964117,7fd4795f-1eea-447b-af0d-ef4f77300e68,e8238cce-f563-424e-b1ee-b9aa967b2549,0.380411,2026-11-19T08:29:50.842346,2026-11-19T08:29:50.842346,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.07081908465725398, ""Credi...",Cash Stack v2 Models,4260009d-b949-43d8-8fb2-74118d07e009,2026-11-19T08:29:50.842346,{},Trench 1,Android,Dev_Train,2026-01-03 08:51:06
2,3678315,48a382c4-ca53-4cf4-9db2-99f96240df39,5cb525ea-d617-40c3-a2cc-c677e9ff2f8e,0.182036,2026-11-19T08:29:50.843584,2026-11-19T08:29:50.843584,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.04154027137957607, ""Credi...",Cash Stack v2 Models,8f22e987-3eeb-4d8a-b30d-d77b5b8ff6c7,2026-11-19T08:29:50.843584,{},Trench 1,Android,Pre_Train,2025-09-11 17:51:49
3,3971836,528f25cb-bfb3-4dd3-a83c-fd86d5da25be,387ca9a6-9718-4744-ac43-cab66ae051f4,0.195505,2026-11-19T08:29:50.844792,2026-11-19T08:29:50.844792,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.16296872322497327, ""Credi...",Cash Stack v2 Models,ffe1f6ab-5c5a-4b56-8d3f-3a0df446021e,2026-11-19T08:29:50.844792,{},Trench 1,Android,Dev_Train,2026-01-05 05:33:33
4,4530659,9e212223-204d-4c7c-9a06-9cd1d0f30803,367b0343-5942-4d2e-a5bf-bc556922bca4,0.213253,2026-11-19T08:29:50.846048,2026-11-19T08:29:50.846048,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.17988531443114764, ""Credi...",Cash Stack v2 Models,58b4ae05-275d-436e-9668-b84d4bf95dca,2026-11-19T08:29:50.846048,{},Trench 1,iOS,Dev_Test,2026-08-11 10:01:20


In [27]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 1,2226,2026-05-01 02:04:31,2026-09-25 20:27:17
1,Dev_Test,iOS,v2,Trench 1,2095,2026-05-01 06:29:58,2026-09-25 12:31:04
2,Dev_Train,Android,v2,Trench 1,9607,2026-01-01 01:30:19,2026-04-30 22:57:47
3,Dev_Train,iOS,v2,Trench 1,4306,2026-01-01 02:15:16,2026-04-30 22:40:24
4,Pre_Train,Android,v2,Trench 1,10076,2025-01-01 00:48:37,2025-12-30 23:25:08
5,Pre_Train,iOS,v2,Trench 1,9250,2025-01-01 03:29:57,2025-12-30 23:04:06


In [28]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=073c61e4-32e9-4918-9ded-5a0f382e4542>

### ₸ Trench 2

#### ₫ Delete

In [29]:
sq = f"""
DELETE FROM {table_id}
WHERE modelDisplayName = 'Alpha-Cash-CIC-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")



Deleted 99302 rows.


In [30]:
sq = """
with base as 
(SELECT
  lmt.customerId as customer_id,
  r.`digitalLoanAccountId`,
  r.`target_fspd30_flag`,
  `dt_loan_appln_dtime` as application_date,
  alt.`ln_disb_dtime`,
  alt.`ln_loan_type`,
  alt.`ln_loan_level_user_type`,
  CASE
        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) <= 30
        THEN 'Trench 1'

        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) > 30
        THEN 'Trench 2'

        WHEN alt.ln_loan_level_user_type = '1_Repeat Applicant'
        THEN 'Trench 3'

        ELSE NULL
    END AS trench_category,
  r.`year_month`,
  r.`ln_os_type` as osType,
  `ScoreRange_WOE`,
  `CreditAvgCreditLimit`,
  `cnt_active_contracts_Credit_Lines`,
  `vel_contract_nongranted_cnt_6on12`,
  `cnt_active_contracts_Credit_Cards`,
  `avg_amt_granted_24M`,
  `cic_total_overdue_to_total_outstanding_ratio`,
  `cic_total_overdue_to_monthly_income_ratio`,
  `CreditMaxOverdueDays`,
  `cic_seq_share_credit_lines_24m`,
  `Credit_Lines_active_contracts_util`,
  `Personal_Loans_granted_contracts_cnt_6M`,
  `has_ever_been_overdue`,
  `cic_seq_cnt_credit_cards_24m`,
  `has_ever_had_risky_contract`,
  `creditcard_utilization_ever`,
  `cic_seq_first_loan_type_24m`,
  `total_overdue_granted_contracts`,
  `creditcard_utilization_combined`,
  `cic_score`,
  case when date(r.dt_loan_appln_dtime) >= '2026-01-01' and date(r.dt_loan_appln_dtime) <= '2026-04-30' then 'Dev_Train'
           when date(r.dt_loan_appln_dtime) > '2026-04-30' and  date(r.dt_loan_appln_dtime) < '2026-10-30'then 'Dev_Test'
           when date(r.dt_loan_appln_dtime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20250101_20261027_score_range_fix` r
Left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
Left join `prj-prod-dataplatform.risk_mart.applied_loans_trans` alt on alt.digitalLoanAccountId = r.digitalLoanAccountId
where  r.`cic_score` is not null 
)
select * from base where trench_category = 'Trench 2'
and  date(application_date) < '2026-10-30' 
and target_fspd30_flag is not null 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 81c44661-174d-4187-b4b9-bb57861dd9c5 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (24988, 31)


In [31]:
feature_column = ['ScoreRange_WOE', 'CreditAvgCreditLimit',   'cnt_active_contracts_Credit_Lines', 'vel_contract_nongranted_cnt_6on12', 'cnt_active_contracts_Credit_Cards', 'avg_amt_granted_24M', 'cic_total_overdue_to_total_outstanding_ratio',
       'cic_total_overdue_to_monthly_income_ratio', 'CreditMaxOverdueDays',  'cic_seq_share_credit_lines_24m', 'Credit_Lines_active_contracts_util', 'Personal_Loans_granted_contracts_cnt_6M', 'has_ever_been_overdue',
       'cic_seq_cnt_credit_cards_24m', 'has_ever_had_risky_contract', 'creditcard_utilization_ever', 'cic_seq_first_loan_type_24m',  'total_overdue_granted_contracts', 'creditcard_utilization_combined', 'cic_score']

dfd = transform_datav2(data, feature_column, a='cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 2', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3068323,ad947c6c-288a-4f2d-a921-15116daf242c,e1d8bec1-04b4-49ab-be01-bca06018c01b,0.220049,2026-11-19T08:30:28.476322,2026-11-19T08:30:28.476322,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.17107752314251712, ""Credi...",Cash Stack v2 Models,3ff68e9f-d789-4c32-b709-48048e83a3b6,2026-11-19T08:30:28.476322,{},Trench 2,Android,Pre_Train,2025-12-10 15:52:45
1,3858416,8cafa3af-1d5f-44f5-8e53-d990d0ec53ad,0fe51dd4-e912-4a31-93ea-9fe85f01596f,0.241925,2026-11-19T08:30:28.478499,2026-11-19T08:30:28.478499,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.27556667808060925, ""Credi...",Cash Stack v2 Models,4c483ea5-d1ca-484a-b892-5d97c88ca532,2026-11-19T08:30:28.478499,{},Trench 2,Android,Dev_Train,2026-02-17 15:24:59
2,3656256,220f34d1-5aa9-43ba-92d3-2c6c901d347b,808bc944-de70-406d-8b87-56aa90ef6fcd,0.165416,2026-11-19T08:30:28.478499,2026-11-19T08:30:28.478499,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.3680418285796078, ""Credit...",Cash Stack v2 Models,953776d0-83d8-43fd-8fb4-932f3e2a7cf3,2026-11-19T08:30:28.478499,{},Trench 2,Android,Pre_Train,2025-12-05 07:43:51
3,2732162,5acc5c56-301c-4429-9974-e69a41832d8a,70c8e3c4-b665-4455-aae7-8378b4fa1517,0.253553,2026-11-19T08:30:28.480791,2026-11-19T08:30:28.480791,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": -0.16247818152180743, ""Cred...",Cash Stack v2 Models,de96f068-1692-4b76-8035-31a0f9020a8c,2026-11-19T08:30:28.480791,{},Trench 2,Android,Dev_Train,2026-03-17 15:43:55
4,3660326,99a33432-66d1-4f24-96f7-07aac6b4cd7b,c9692443-dda1-4d9a-a2dd-9af481f28b35,0.208626,2026-11-19T08:30:28.481519,2026-11-19T08:30:28.481519,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": -0.27448585216942006, ""Cred...",Cash Stack v2 Models,3e045159-50cb-4008-89de-991924455439,2026-11-19T08:30:28.481519,{},Trench 2,Android,Dev_Train,2026-02-11 10:57:38


In [32]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 2,1451,2026-05-01 07:45:58,2026-09-25 20:35:51
1,Dev_Test,iOS,v2,Trench 2,1001,2026-05-01 01:43:33,2026-09-23 12:24:19
2,Dev_Train,Android,v2,Trench 2,3690,2026-01-01 01:24:57,2026-04-30 23:18:18
3,Dev_Train,iOS,v2,Trench 2,2628,2026-01-01 08:51:47,2026-04-30 19:15:33
4,Pre_Train,Android,v2,Trench 2,8586,2025-01-01 01:14:33,2025-12-30 23:07:21
5,Pre_Train,iOS,v2,Trench 2,7632,2025-01-01 07:42:43,2025-12-30 22:32:02


In [33]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e7cd67c1-be3e-4094-b443-1a8f8f918324>

### ₸ Trench 3

#### ₫ Delete

In [34]:
sq = f"""
DELETE FROM {table_id}
WHERE modelDisplayName = 'Alpha-Cash-CIC-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")



Deleted 18945 rows.


In [35]:
sq = """
with base as 
(SELECT
  lmt.customerId as customer_id,
  r.`digitalLoanAccountId`,
  r.`target_fspd30_flag`,
  `dt_loan_appln_dtime` as application_date,
  alt.`ln_disb_dtime`,
  alt.`ln_loan_type`,
  alt.`ln_loan_level_user_type`,
  CASE
        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) <= 30
        THEN 'Trench 1'

        WHEN alt.ln_loan_level_user_type = '2_New Applicant'
             AND DATE_DIFF(
                 DATE(ln_appln_submit_datetime),
                 DATE(onb_tsa_onboarding_datetime),
                 DAY
             ) > 30
        THEN 'Trench 2'

        WHEN alt.ln_loan_level_user_type = '1_Repeat Applicant'
        THEN 'Trench 3'

        ELSE NULL
    END AS trench_category,
  r.`year_month`,
  r.`ln_os_type` as osType,
  `ScoreRange_WOE`,
  `CreditAvgCreditLimit`,
  `cnt_active_contracts_Credit_Lines`,
  `vel_contract_nongranted_cnt_6on12`,
  `cnt_active_contracts_Credit_Cards`,
  `avg_amt_granted_24M`,
  `cic_total_overdue_to_total_outstanding_ratio`,
  `cic_total_overdue_to_monthly_income_ratio`,
  `CreditMaxOverdueDays`,
  `cic_seq_share_credit_lines_24m`,
  `Credit_Lines_active_contracts_util`,
  `Personal_Loans_granted_contracts_cnt_6M`,
  `has_ever_been_overdue`,
  `cic_seq_cnt_credit_cards_24m`,
  `has_ever_had_risky_contract`,
  `creditcard_utilization_ever`,
  `cic_seq_first_loan_type_24m`,
  `total_overdue_granted_contracts`,
  `creditcard_utilization_combined`,
  `cic_score`,
  case when date(r.dt_loan_appln_dtime) >= '2026-01-01' and date(r.dt_loan_appln_dtime) <= '2026-04-30' then 'Dev_Train'
           when date(r.dt_loan_appln_dtime) > '2026-04-30' and  date(r.dt_loan_appln_dtime) < '2026-10-30'then 'Dev_Test'
           when date(r.dt_loan_appln_dtime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20250101_20261027_score_range_fix` r
Left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
Left join `prj-prod-dataplatform.risk_mart.applied_loans_trans` alt on alt.digitalLoanAccountId = r.digitalLoanAccountId
where  r.`cic_score` is not null 
)
select * from base where trench_category = 'Trench 3'
and  date(application_date) < '2026-10-30' 
and target_fspd30_flag is not null 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID f6dea2cd-c3ff-4d41-a3a9-ef850025807d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (18945, 31)


In [36]:
feature_column = ['ScoreRange_WOE', 'CreditAvgCreditLimit',   'cnt_active_contracts_Credit_Lines', 'vel_contract_nongranted_cnt_6on12', 'cnt_active_contracts_Credit_Cards', 'avg_amt_granted_24M', 'cic_total_overdue_to_total_outstanding_ratio',
       'cic_total_overdue_to_monthly_income_ratio', 'CreditMaxOverdueDays',  'cic_seq_share_credit_lines_24m', 'Credit_Lines_active_contracts_util', 'Personal_Loans_granted_contracts_cnt_6M', 'has_ever_been_overdue',
       'cic_seq_cnt_credit_cards_24m', 'has_ever_had_risky_contract', 'creditcard_utilization_ever', 'cic_seq_first_loan_type_24m',  'total_overdue_granted_contracts', 'creditcard_utilization_combined', 'cic_score']

dfd = transform_datav2(data, feature_column, a='cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 3', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3347747,ce353a93-0609-4e5f-b04d-5f2374be3851,ed1fc2a4-19fa-46e4-a6d6-9a8919664585,0.164675,2026-11-19T08:30:52.958727,2026-11-19T08:30:52.958727,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.16296872322497327, ""Credi...",Cash Stack v2 Models,f80d086e-4598-441a-ba92-a6244ac03357,2026-11-19T08:30:52.958727,{},Trench 3,Android,Pre_Train,2025-12-14 21:45:17
1,1968769,26145978-c070-4ce6-8dee-adb1e7468363,9fbd2ba3-5dc2-46fe-a27a-c5db3989bf41,0.106978,2026-11-19T08:30:52.959720,2026-11-19T08:30:52.959720,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": -0.09420533286524543, ""Cred...",Cash Stack v2 Models,b1939d3b-2d5c-4531-8d5c-5ffeb5b9a61a,2026-11-19T08:30:52.959720,{},Trench 3,Android,Pre_Train,2025-11-17 15:23:54
2,4309865,293108bb-9dcd-451b-8b41-b17bbf81ae46,82b89abf-e30a-44cd-9711-3140b15bca52,0.316235,2026-11-19T08:30:52.960723,2026-11-19T08:30:52.960723,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": -0.27448585216942006, ""Cred...",Cash Stack v2 Models,452491f9-92e3-452c-b4d8-47a12886e3a4,2026-11-19T08:30:52.960723,{},Trench 3,iOS,Dev_Test,2026-09-02 16:01:32
3,3101075,a6240a66-a0bf-4755-abe7-741ef76e4970,7f0818a8-0359-4109-8f62-1f5812fe76ff,0.174375,2026-11-19T08:30:52.960723,2026-11-19T08:30:52.960723,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.3680418285796078, ""Credit...",Cash Stack v2 Models,31b45edd-2ca2-419e-8331-104e40a3db7a,2026-11-19T08:30:52.960723,{},Trench 3,iOS,Dev_Test,2026-08-20 11:40:33
4,3697742,25ab48b6-6663-4363-9f41-f4c5228e7af4,645a8b74-2414-4535-8631-5c803a61410f,0.099960,2026-11-19T08:30:52.961721,2026-11-19T08:30:52.961721,Alpha-Cash-CIC-Model,v2,"{""ScoreRange_WOE"": 0.16296872322497327, ""Credi...",Cash Stack v2 Models,8a578f76-21a4-4660-8e50-ed7edaf0bbe4,2026-11-19T08:30:52.961721,{},Trench 3,iOS,Dev_Test,2026-05-25 17:02:38


In [37]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 3,1892,2026-05-01 09:52:47,2026-09-25 21:39:33
1,Dev_Test,iOS,v2,Trench 3,1886,2026-05-01 00:25:51,2026-09-25 07:20:44
2,Dev_Train,Android,v2,Trench 3,1846,2026-01-01 09:25:56,2026-04-30 22:14:13
3,Dev_Train,iOS,v2,Trench 3,1949,2026-01-01 07:39:26,2026-04-30 21:37:02
4,Pre_Train,Android,v2,Trench 3,5793,2025-01-01 08:20:48,2025-12-30 20:23:36
5,Pre_Train,iOS,v2,Trench 3,5579,2025-01-01 02:13:15,2025-12-30 22:21:21


In [38]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=80ff54f2-426b-4c79-bee6-9f6d96eb7c03>

# 🪦💀 Graveyard

#### PSI Functions new

In [ ]:
# ## Updated on 27-10-2025 - Modified for Training Period Baseline
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Tuple
# import warnings
# warnings.filterwarnings('ignore')

# def identify_feature_types(df: pd.DataFrame, feature_list: List[str]) -> Dict[str, List[str]]:
#     """
#     Identify categorical and numerical features from the feature list.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to classify

#     Returns:
#     --------
#     Dict with 'categorical' and 'numerical' keys containing respective feature lists
#     """
#     categorical_features = []
#     numerical_features = []

#     for feature in feature_list:
#         if feature not in df.columns:
#             print(f"Warning: Feature '{feature}' not found in dataframe")
#             continue

#         # Check if feature is numeric
#         if pd.api.types.is_numeric_dtype(df[feature]):
#             # If unique values are less than 15 and all integers, treat as categorical
#             unique_vals = df[feature].nunique()
#             if unique_vals < 15 and df[feature].dropna().apply(lambda x: x == int(x) if isinstance(x, (int, float)) else False).all():
#                 categorical_features.append(feature)
#             else:
#                 numerical_features.append(feature)
#         else:
#             categorical_features.append(feature)

#     return {
#         'categorical': categorical_features,
#         'numerical': numerical_features
#     }


# def create_bins_for_features(df: pd.DataFrame,
#                              numerical_features: List[str],
#                              categorical_features: List[str],
#                              train_period_df: pd.DataFrame) -> Dict:
#     """
#     Create bins for numerical features (deciles with fallback) and categorical features (top 6 + others)
#     based on the entire training period data.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Full input dataframe
#     numerical_features : List[str]
#         List of numerical features
#     categorical_features : List[str]
#         List of categorical features
#     train_period_df : pd.DataFrame
#         Training period dataframe (June 2024 to March 2025)

#     Returns:
#     --------
#     Dictionary containing binning information for each feature
#     """
#     binning_info = {}

#     # Create bins for numerical features with fallback strategy
#     for feature in numerical_features:
#         valid_data = train_period_df[feature].dropna()

#         if len(valid_data) == 0:
#             binning_info[feature] = {'type': 'numerical', 'bins': None, 'bin_ranges': {}}
#             continue

#         bins = None
#         bin_count = None

#         # Try 10 bins (deciles)
#         try:
#             test_bins = np.percentile(valid_data, np.arange(0, 101, 10))
#             test_bins = np.unique(test_bins)
#             if len(test_bins) >= 11:  # 11 edges = 10 bins
#                 bins = test_bins
#                 bin_count = 10
#         except Exception as e:
#             pass

#         # If 10 bins not possible, try 5 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, np.arange(0, 101, 20))
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 6:  # 6 edges = 5 bins
#                     bins = test_bins
#                     bin_count = 5
#             except Exception as e:
#                 pass

#         # If 5 bins not possible, try 3 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, [0, 33.33, 66.67, 100])
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 4:  # 4 edges = 3 bins
#                     bins = test_bins
#                     bin_count = 3
#             except Exception as e:
#                 pass

#         # If still no bins possible, use equal distance bins of 5
#         if bins is None:
#             print(f"Warning: Feature '{feature}' has insufficient variance - cannot create standard bins")
#             print(f"Feature '{feature}': Using equal distance bins of 5")

#             min_val = valid_data.min()
#             max_val = valid_data.max()

#             # Create 5 equal distance bins
#             bins = np.linspace(min_val, max_val, 6)  # 6 edges = 5 bins
#             bins = np.unique(bins)
#             bin_count = len(bins) - 1

#             # If all values are the same, add slight buffer
#             if bin_count == 1:
#                 bins = np.array([min_val - 0.1, min_val, min_val + 0.1])
#                 bin_count = 2
#                 print(f"Feature '{feature}': Constant value ({min_val}). Created 2 equal distance bins with buffer")

#         # Add infinity edges to capture all values
#         bins = bins.copy()
#         bins[0] = -np.inf
#         bins[-1] = np.inf

#         print(f"Feature '{feature}': Created {bin_count} bins")

#         # Create bin ranges dictionary
#         bin_ranges = {}
#         for i in range(len(bins)-1):
#             bin_name = f"Bin_{i+1}"
#             bin_ranges[bin_name] = {
#                 'min': bins[i],
#                 'max': bins[i+1],
#                 'range_str': f"[{bins[i]:.2f}, {bins[i+1]:.2f}]" if not np.isinf(bins[i]) and not np.isinf(bins[i+1]) else f"({bins[i]}, {bins[i+1]})"
#             }

#         binning_info[feature] = {
#             'type': 'numerical',
#             'bins': bins,
#             'bin_ranges': bin_ranges,
#             'bin_count': bin_count
#         }

#     # Create bins for categorical features (top 6 + others) using training period
#     for feature in categorical_features:
#         value_counts = train_period_df[feature].value_counts()
#         unique_categories = value_counts.index.tolist()
#         print(f"Unique categories: {unique_categories}")

#         if len(unique_categories) <= 6:
#             # Treat each category as a separate bin
#             top_categories = unique_categories
#         else:
#             # Use top 6 categories only
#             top_categories = value_counts.nlargest(6).index.tolist()

#         print(f"Top categories for feature '{feature}': {top_categories}")

#         binning_info[feature] = {
#                 'type': 'categorical',
#                 'top_categories': top_categories,
#                 'bin_ranges': {}  # No ranges for categorical
#             }

#     return binning_info


# def apply_binning(df: pd.DataFrame,
#                   feature: str,
#                   binning_info: Dict) -> pd.Series:
#     """
#     Apply binning to a feature based on binning information.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature : str
#         Feature name
#     binning_info : Dict
#         Binning information for the feature

#     Returns:
#     --------
#     pd.Series with binned values
#     """
#     if binning_info['type'] == 'numerical':
#         if binning_info['bins'] is None:
#             return pd.Series(['Missing'] * len(df), index=df.index)

#         bins = binning_info['bins']
#         labels = [f"Bin_{i+1}" for i in range(len(bins)-1)]

#         binned = pd.cut(df[feature],
#                        bins=bins,
#                        labels=labels,
#                        include_lowest=True,
#                        duplicates='drop')

#         # Handle nulls - convert to string and then replace
#         binned = binned.astype(str)
#         binned[df[feature].isna()] = 'Missing'

#         return binned

#     else:  # categorical
#         top_cats = binning_info['top_categories']

#         # Convert to string for consistent comparison
#         if pd.api.types.is_categorical_dtype(df[feature]):
#             feature_data = df[feature].astype(str)
#         else:
#             feature_data = df[feature].astype(str)

#         # Replace NaN string representation with 'Missing'
#         feature_data = feature_data.replace('nan', 'Missing')

#         # Convert top_cats to strings for comparison
#         top_cats_str = [str(cat) for cat in top_cats]

#         # Apply binning logic: use category name if in top_cats, else 'Others' (except for Missing)
#         binned = feature_data.apply(lambda x: x if x in top_cats_str else ('Others' if x != 'Missing' else 'Missing'))

#         return binned


# def calculate_psi(expected_pct: pd.Series,
#                   actual_pct: pd.Series,
#                   epsilon: float = 0.0001) -> float:
#     """
#     Calculate Population Stability Index with proper epsilon handling and renormalization.

#     Parameters:
#     -----------
#     expected_pct : pd.Series
#         Expected (baseline) percentages
#     actual_pct : pd.Series
#         Actual percentages
#     epsilon : float
#         Small value to avoid log(0)

#     Returns:
#     --------
#     PSI value
#     """
#     # Align indices
#     all_bins = expected_pct.index.union(actual_pct.index)
#     expected_pct = expected_pct.reindex(all_bins, fill_value=0)
#     actual_pct = actual_pct.reindex(all_bins, fill_value=0)

#     # Only add epsilon where values are zero
#     expected_pct = expected_pct.apply(lambda x: epsilon if x == 0 else x)
#     actual_pct = actual_pct.apply(lambda x: epsilon if x == 0 else x)

#     # Renormalize to ensure they sum to 1 after adding epsilon
#     expected_pct = expected_pct / expected_pct.sum()
#     actual_pct = actual_pct / actual_pct.sum()

#     # Calculate PSI
#     psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

#     return psi_value


# def calculate_month_on_month_psi(df: pd.DataFrame,
#                                  feature_list: List[str],
#                                  segment_columns: List[str],
#                                  month_col: str = 'Application_month',
#                                  data_selection_col: str = 'Data_selection',
#                                  account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate PSI for each feature comparing training period (June 2024 to March 2025)
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column (identifies train period)
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with PSI values with one row per feature-month-segment combination
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []

#     # Calculate overall PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(f"Feature binned {df[f'{feature}_binned']}")
#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate PSI for each test month
#         for month in test_months:
#             actual_dist = df[df[month_col] == month][f'{feature}_binned'].value_counts(normalize=True)
#             psi_value = calculate_psi(train_baseline, actual_dist)

#             # Calculate average percentages across all bins
#             expected_avg_pct = train_baseline.mean() * 100
#             actual_avg_pct = actual_dist.mean() * 100

#             # # Count distinct accounts for segment
#             # base_segment_count = train_segment[account_id_col].nunique()
#             # actual_segment_count = actual_segment[account_id_col].nunique()


#             results.append({
#                 'Feature': feature,
#                 'Feature_Type': binning_info[feature]['type'],
#                 'Segment_Column': 'Overall',
#                 'Segment_Value': 'All',
#                 'Month': f"{month}",
#                 'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                 'Current_Month': month,
#                 'Expected_Percentage': expected_avg_pct,
#                 'Actual_Percentage': actual_avg_pct,
#                 'PSI': psi_value
#             })

#     # Calculate PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)
#                     psi_value = calculate_psi(train_baseline, actual_dist)

#                     # Calculate average percentages across all bins
#                     expected_avg_pct = train_baseline.mean() * 100
#                     actual_avg_pct = actual_dist.mean() * 100

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     results.append({
#                         'Feature': feature,
#                         'Feature_Type': binning_info[feature]['type'],
#                         'Segment_Column': segment_col,
#                         'Segment_Value': segment_val,
#                         'Month': f"{month}",
#                         'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                         'Current_Month': month,
#                         'Base_Count': base_segment_count,
#                         'Actual_Count': actual_segment_count,
#                         'Expected_Percentage': expected_avg_pct,
#                         'Actual_Percentage': actual_avg_pct,
#                         'PSI': psi_value
#                     })

#     return pd.DataFrame(results)


# def calculate_bin_level_psi(df: pd.DataFrame,
#                             feature_list: List[str],
#                             segment_columns: List[str],
#                             month_col: str = 'Application_month',
#                             data_selection_col: str = 'Data_selection',
#                             account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate bin-level PSI for each feature comparing training period
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with bin-level PSI details including bin ranges
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []
#     epsilon = 0.0001

#     # Calculate overall bin-level PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(df[f'{feature}_binned'])

#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate bin-level PSI for each test month
#         for month in test_months:
#             month_data = df[df[month_col] == month]
#             actual_dist = month_data[f'{feature}_binned'].value_counts(normalize=True)

#             # Count distinct accounts
#             base_count = df[df[data_selection_col] == 'Train'][account_id_col].nunique()
#             actual_count = month_data[account_id_col].nunique()

#             # Get all bins
#             all_bins = train_baseline.index.union(actual_dist.index)

#             for bin_name in all_bins:
#                 # Simplified epsilon logic - no redundancy
#                 expected_pct = train_baseline.get(bin_name, 0)
#                 actual_pct = actual_dist.get(bin_name, 0)

#                 # Add epsilon only if zero
#                 expected_pct = epsilon if expected_pct == 0 else expected_pct
#                 actual_pct = epsilon if actual_pct == 0 else actual_pct

#                 # Calculate bin-level PSI
#                 bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                 # Get bin range information
#                 bin_ranges = binning_info[feature]['bin_ranges']
#                 if bin_name in bin_ranges:
#                     bin_min = bin_ranges[bin_name]['min']
#                     bin_max = bin_ranges[bin_name]['max']
#                     bin_range = bin_ranges[bin_name]['range_str']
#                 else:
#                     # For categorical or special bins (Missing, Others)
#                     bin_min = None
#                     bin_max = None
#                     bin_range = bin_name

#                 results.append({
#                     'Feature': feature,
#                     'Feature_Type': binning_info[feature]['type'],
#                     'Segment_Column': 'Overall',
#                     'Segment_Value': 'All',
#                     'Month': f"{month}",
#                     'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                     'Current_Month': month,
#                     'Base_Count': base_count,
#                     'Actual_Count': actual_count,
#                     'Bin': bin_name,
#                     'Bin_Range': bin_range,
#                     'Bin_Min': bin_min,
#                     'Bin_Max': bin_max,
#                     'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                     'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                     'Bin_PSI': bin_psi
#                 })

#     # Calculate bin-level PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate bin-level PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     # Get all bins
#                     all_bins = train_baseline.index.union(actual_dist.index)

#                     for bin_name in all_bins:
#                         # Simplified epsilon logic - no redundancy
#                         expected_pct = train_baseline.get(bin_name, 0)
#                         actual_pct = actual_dist.get(bin_name, 0)

#                         # Add epsilon only if zero
#                         expected_pct = epsilon if expected_pct == 0 else expected_pct
#                         actual_pct = epsilon if actual_pct == 0 else actual_pct

#                         # Calculate bin-level PSI
#                         bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                         # Get bin range information
#                         bin_ranges = binning_info[feature]['bin_ranges']
#                         if bin_name in bin_ranges:
#                             bin_min = bin_ranges[bin_name]['min']
#                             bin_max = bin_ranges[bin_name]['max']
#                             bin_range = bin_ranges[bin_name]['range_str']
#                         else:
#                             # For categorical or special bins (Missing, Others)
#                             bin_min = None
#                             bin_max = None
#                             bin_range = bin_name

#                         results.append({
#                             'Feature': feature,
#                             'Feature_Type': binning_info[feature]['type'],
#                             'Segment_Column': segment_col,
#                             'Segment_Value': segment_val,
#                             'Month': f"{month}",
#                             'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                             'Current_Month': month,
#                             'Base_Count': base_segment_count,
#                             'Actual_Count': actual_segment_count,
#                             'Bin': bin_name,
#                             'Bin_Range': bin_range,
#                             'Bin_Min': bin_min,
#                             'Bin_Max': bin_max,
#                             'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                             'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                             'Bin_PSI': bin_psi
#                         })

#     return pd.DataFrame(results)

# End